## 추론

In [22]:
import pandas as pd
import os
import re
import json
import yaml
from datetime import datetime
from glob import glob
from tqdm import tqdm
from pprint import pprint
import torch
import pytorch_lightning as pl
from rouge import Rouge # 모델의 성능을 평가하기 위한 라이브러리입니다.

from torch.utils.data import Dataset , DataLoader
from transformers import AutoTokenizer, BartForConditionalGeneration, BartConfig
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import Trainer, TrainingArguments
from transformers import EarlyStoppingCallback
from transformers import AutoConfig, AutoModelForSeq2SeqLM, AutoTokenizer


import wandb # 모델 학습 과정을 손쉽게 Tracking하고, 시각화할 수 있는 라이브러리입니다.

In [23]:
# 가장 최신의 run name 가져옴
run_name = sorted(os.listdir("./checkpoint"))[-1]
print(f"✅ 자동 선택된 최신 run name: {run_name}")

✅ 자동 선택된 최신 run name: t5_base_korean_text_summary_v1_1204_0422


In [24]:
# best checkpoint 자동 선택 함수

def auto_select_best_checkpoint(output_dir="./"):
    # 1. 모든 체크포인트 폴더 가져오기
    # (폴더명 예시: checkpoint-500, checkpoint-1000 ...)
    checkpoints = glob(os.path.join(output_dir, "checkpoint-*"))
    
    best_loss = float('inf')
    best_step = 0
    best_ckpt_path = ""
    
    print(checkpoints)
    print(f"🕵️‍♂️ 분석 시작: 총 {len(checkpoints)}개의 체크포인트를 검사합니다.")
    print("-" * 50)

    for ckpt in checkpoints:
        # 각 폴더 안의 성적표(trainer_state.json) 확인
        state_file = os.path.join(ckpt, "trainer_state.json")
        
        if os.path.exists(state_file):
            try:
                with open(state_file, 'r') as f:
                    data = json.load(f)
                    
                    # 로그 기록 중 'eval_loss'가 있는 가장 마지막 기록 확인
                    # (보통 체크포인트 저장 시점의 평가 점수임)
                    for log in reversed(data['log_history']):
                        if 'eval_loss' in log:
                            loss = log['eval_loss']
                            step = log['step']
                            
                            print(f"  📂 Step {step}: Loss {loss:.4f}")
                            
                            # 1등 갱신!
                            if loss < best_loss:
                                best_loss = loss
                                best_step = step
                                best_ckpt_path = ckpt
                            break 
            except:
                pass # 파일 깨졌으면 패스
    
    print("-" * 50)
    if best_ckpt_path:
        print(f"🏆 [최종 우승] Step {best_step} (Loss: {best_loss:.5f})")
        print(f"   -> 경로: {best_ckpt_path}")
        return best_ckpt_path
    else:
        print("🚨 성적표를 찾을 수 없습니다.")
        return None

In [25]:
# run_name 하위 체크포인트 디렉토리 설정
output_dir = f"./checkpoint/{run_name}/"

best_path = auto_select_best_checkpoint(output_dir)
print(f"✅ 자동 선택된 최고 체크포인트 경로: {best_path}")

['./checkpoint/t5_base_korean_text_summary_v1_1204_0422/checkpoint-389', './checkpoint/t5_base_korean_text_summary_v1_1204_0422/checkpoint-583', './checkpoint/t5_base_korean_text_summary_v1_1204_0422/checkpoint-194']
🕵️‍♂️ 분석 시작: 총 3개의 체크포인트를 검사합니다.
--------------------------------------------------
  📂 Step 389: Loss 1.6863
  📂 Step 583: Loss 1.5957
  📂 Step 194: Loss 1.9300
--------------------------------------------------
🏆 [최종 우승] Step 583 (Loss: 1.59565)
   -> 경로: ./checkpoint/t5_base_korean_text_summary_v1_1204_0422/checkpoint-583
✅ 자동 선택된 최고 체크포인트 경로: ./checkpoint/t5_base_korean_text_summary_v1_1204_0422/checkpoint-583


In [26]:
# 저장된 config 파일을 불러옵니다.
config_path = "./config_lcw99.yaml"

with open(config_path, "r") as file:
    loaded_config = yaml.safe_load(file)

# 불러온 config 파일의 전체 내용을 확인합니다.
pprint(loaded_config)

{'general': {'data_path': '../../../../data/processed',
             'model_name': 'lcw99/t5-base-korean-text-summary',
             'output_dir': './'},
 'inference': {'batch_size': 16,
               'ckt_path': './checkpoint/',
               'early_stopping': True,
               'generate_max_length': 100,
               'no_repeat_ngram_size': 2,
               'num_beams': 4,
               'remove_tokens': ['</s>', '<pad>'],
               'result_path': './prediction/'},
 'tokenizer': {'decoder_max_len': 128,
               'encoder_max_len': 512,
               'eos_token': '</s>',
               'special_tokens': ['#Person1#',
                                  '#Person2#',
                                  '#Person3#',
                                  '#Person4#',
                                  '#Person5#',
                                  '#Person6#',
                                  '#Person7#',
                                  '#PhoneNumber#',
                     

In [27]:
loaded_config['inference']['ckt_path'] = best_path

In [28]:
# Test에 사용되는 Dataset 클래스를 정의합니다.
class DatasetForInference(Dataset):
    def __init__(self, encoder_input, test_id, len):
        self.encoder_input = encoder_input
        self.test_id = test_id
        self.len = len

    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encoder_input.items()}
        item['ID'] = self.test_id[idx]
        return item

    def __len__(self):
        return self.len


In [29]:
# tokenization 과정까지 진행된 최종적으로 모델에 입력될 데이터를 출력합니다.
def prepare_test_dataset(config,preprocessor, tokenizer):

    test_file_path = os.path.join(config['general']['data_path'],'test_preprocessed.csv')
    print(test_file_path)
    test_data = preprocessor.make_set_as_df(test_file_path,is_train=False)
    test_id = test_data['fname']

    print('-'*150)
    print(f'test_data:\n{test_data["dialogue"][0]}')
    print('-'*150)

    encoder_input_test, _ = preprocessor.make_input(test_data, is_test=True)
    print('-'*10, 'Load data complete', '-'*10,)

    test_tokenized_encoder_inputs = tokenizer(encoder_input_test, return_tensors="pt", padding=True,
                    add_special_tokens=True, truncation=True, max_length=config['tokenizer']['encoder_max_len'], return_token_type_ids=False,)

    test_encoder_inputs_dataset = DatasetForInference(test_tokenized_encoder_inputs, test_id, len(encoder_input_test))
    print('-'*10, 'Make dataset complete', '-'*10,)

    return test_data, test_encoder_inputs_dataset

In [30]:
# 추론을 위한 tokenizer와 학습시킨 모델을 불러옵니다.
def load_tokenizer_and_model_for_test(config,device):
    print('-'*10, 'Load tokenizer & model', '-'*10,)

    model_name = config['general']['model_name']
    ckt_path = config['inference']['ckt_path']
    print('-'*10, f'Model Name : {model_name}', '-'*10,)
    # tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained("paust/pko-t5-base")

    special_tokens_dict = {'additional_special_tokens': config['tokenizer']['special_tokens']}
    tokenizer.add_special_tokens(special_tokens_dict)

    generate_model = AutoModelForSeq2SeqLM.from_pretrained(ckt_path)
    generate_model.resize_token_embeddings(len(tokenizer))
    generate_model.to(device)
    
    print('-'*10, 'Load tokenizer & model complete', '-'*10,)

    return generate_model , tokenizer

In [31]:
# 데이터 전처리를 위한 클래스로, 데이터셋을 데이터프레임으로 변환하고 인코더와 디코더의 입력을 생성합니다.
class Preprocess:
    def __init__(self, eos_token: str) -> None:

        self.eos_token = eos_token

    @staticmethod
    # 실험에 필요한 컬럼을 가져옵니다.
    def make_set_as_df(file_path, is_train = True):
        if is_train:
            df = pd.read_csv(file_path)
            train_df = df[['fname','dialogue','summary']]
            return train_df
        else:
            df = pd.read_csv(file_path)
            test_df = df[['fname','dialogue']]
            return test_df

    # BART 모델의 입력, 출력 형태를 맞추기 위해 전처리를 진행합니다.
    def make_input(self, dataset,is_test = False):
        prefix = "요약: "

        if is_test:
            encoder_input = dataset['dialogue'].apply(lambda x: prefix + str(x))
            return encoder_input.tolist(), None
        else:
            encoder_input = dataset['dialogue'].apply(lambda x: prefix + str(x))
            # T5 학습 시 decoder_input은 정답 그대로 쓰면 됩니다 (EOS만 붙임)
            decoder_input = dataset['summary'].apply(lambda x: str(x)) 
            decoder_output = dataset['summary'].apply(lambda x: str(x) + self.eos_token)
            return encoder_input.tolist(), decoder_input.tolist(), decoder_output.tolist()

In [32]:
# 학습된 모델이 생성한 요약문의 출력 결과를 보여줍니다.
def inference(config):
    device = torch.device('cuda:0' if torch.cuda.is_available()  else 'cpu')
    print('-'*10, f'device : {device}', '-'*10,)
    print(torch.__version__)

    generate_model, tokenizer = load_tokenizer_and_model_for_test(config,device)

    preprocessor = Preprocess(config['tokenizer']['eos_token'])

    test_data, test_encoder_inputs_dataset = prepare_test_dataset(config,preprocessor, tokenizer)
    dataloader = DataLoader(test_encoder_inputs_dataset, batch_size=config['inference']['batch_size'])

    summary = []

    generate_model.eval() # 평가 모드

    with torch.no_grad():
        for item in tqdm(dataloader):
            # T5 Generate
            generated_ids = generate_model.generate(
                input_ids=item['input_ids'].to(device),
                attention_mask=item['attention_mask'].to(device), # attention_mask 추가 (권장)
                no_repeat_ngram_size=config['inference']['no_repeat_ngram_size'],
                early_stopping=config['inference']['early_stopping'],
                max_length=config['inference']['generate_max_length'],
                num_beams=config['inference']['num_beams'],
            )
            
            # 디코딩
            for ids in generated_ids:
                result = tokenizer.decode(ids, skip_special_tokens=True) # 스페셜 토큰 자동 제거
                summary.append(result)

    output = pd.DataFrame(
        {
            "fname": test_data['fname'],
            "summary" : summary,
        }
    )

    result_path = config['inference']['result_path']
    if not os.path.exists(result_path):
        os.makedirs(result_path)
    
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"output_{timestamp}.csv"
    
    save_path = os.path.join(result_path, filename)
    output.to_csv(save_path, index=False)
    
    print(f"✅ 추론 완료! 저장 경로: {save_path}")
    
    return output

In [34]:
# 학습된 모델의 test를 진행합니다.
if __name__ == "__main__":
    output = inference(loaded_config)

---------- device : cuda:0 ----------
2.1.0
---------- Load tokenizer & model ----------
---------- Model Name : lcw99/t5-base-korean-text-summary ----------
---------- Load tokenizer & model complete ----------
../../../../data/processed/test_preprocessed.csv
------------------------------------------------------------------------------------------------------------------------------------------------------
test_data:
#Person1# Ms. Dawson, 받아쓰기 좀 부탁드려야겠어요. #Person2# 네, 말씀하세요... #Person1# 이걸 오늘 오후까지 모든 직원들에게 사내 메모로 보내야 해요. 준비됐나요? #Person2# 네, 말씀하세요. #Person1# 모든 직원에게 알립니다... 즉시 발효되어 모든 사내 통신은 이메일과 공식 메모로만 제한됩니다. 근무 시간 동안 즉시 메시지 프로그램 사용은 금지됩니다. #Person2# 이 정책이 사내 통신에만 적용되나요, 아니면 외부 통신에도 해당되나요? #Person1# 이는 모든 통신에 적용됩니다. 사무실 내 직원 간 통신 뿐만 아니라 외부 통신도 해당됩니다. #Person2# 하지만 많은 직원들이 고객과 소통하려고 즉시 메시지를 사용합니다. #Person1# 통신 방법을 바꿔야 할 것입니다. 이 사무실에서는 즉시 메시지를 사용하는 것을 원하지 않습니다. 너무 많은 시간이 낭비됩니다! 이제 계속해서 메모를 작성해 주세요. 어디까지 했죠? #Person2# 내외부 통신에 적용됩니다. #Person1# 네. 즉시 메시지를 계속 사용하면 경고 후 시정 조치가 이루어지며, 두 번째 

100%|██████████| 32/32 [01:05<00:00,  2.04s/it]

✅ 추론 완료! 저장 경로: ./prediction/output_20251204_060507.csv
